# Manual Command Surface Validation

Notebook companion for `manual-command-surface-validation.md`.

Aligned through SPINE.27 Hot State CLI + Manual Validation.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess

def find_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "cmd/yai/Cargo.toml").is_file() and (candidate / "docs/manuals").is_dir():
            return candidate
    raise RuntimeError("yai-core root not found")

ROOT = find_root(Path.cwd())
PREFIX = Path("/tmp/yai-core-install-test")
YAI_HOME = Path("/tmp/yai-core-home-test")
YAI_SOCKET = YAI_HOME / "run" / "yaid.sock"
ENV = os.environ.copy()
ENV["YAI_HOME"] = str(YAI_HOME)
ENV["PATH"] = f"{PREFIX / 'bin'}:{ENV.get('PATH', '')}"

def run(cmd: str, check: bool = True) -> str:
    print(f"$ {cmd}")
    proc = subprocess.run(cmd, cwd=ROOT, env=ENV, shell=True, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with {proc.returncode}: {cmd}")
    return proc.stdout

print(ROOT)

## Dev Binary Checks

In [ ]:
run("make build")
run("target/debug/yai --version")
run("target/debug/yai info")
run("target/debug/yai doctor")
run("target/debug/yai hot status", check=False)
run("build/yaid --version")

## Installed Runtime Checks

In [ ]:
run(f"rm -rf {shlex.quote(str(PREFIX))} {shlex.quote(str(YAI_HOME))}")
run(f"make install-local PREFIX={shlex.quote(str(PREFIX))} YAI_HOME={shlex.quote(str(YAI_HOME))}")
run("yai --version")
run("yai info")
run("yai doctor")
run("yai hot status", check=False)
run(f"{shlex.quote(str(PREFIX / 'bin' / 'yaid'))} --version")

## Daemon, Hot State, And Projection Freshness

In [ ]:
run(f"mkdir -p {shlex.quote(str(YAI_HOME / 'run'))}")
daemon = subprocess.Popen(
    [str(PREFIX / "bin" / "yaid"), "--socket", str(YAI_SOCKET), "--foreground"],
    cwd=ROOT,
    env=ENV,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    text=True,
)
try:
    run("sleep 1")
    run(f"yai daemon status --socket {shlex.quote(str(YAI_SOCKET))}")
    run(f"yai daemon info --socket {shlex.quote(str(YAI_SOCKET))}")
    run(f"yai daemon run-minimum-loop --socket {shlex.quote(str(YAI_SOCKET))}")
    run("yai hot status")
    fs_output = run(f"yai daemon run-filesystem-loop --socket {shlex.quote(str(YAI_SOCKET))}")
    run("yai hot status")
    import re
    match = re.search(r'"journal_path":"([^"]+)"', fs_output)
    if not match:
        raise RuntimeError("journal_path not found")
    journal = match.group(1)
    run(f"yai projection inspect --journal {shlex.quote(journal)} --consumer model")
finally:
    run(f"yai daemon shutdown --socket {shlex.quote(str(YAI_SOCKET))}", check=False)
    daemon.wait(timeout=5)

## Snapshot Edge Cases And Uninstall

In [ ]:
run(f"rm -f {shlex.quote(str(YAI_HOME / 'run' / 'hot-state.json'))}")
run("yai hot status", check=False)
run(f"printf '{{broken\\n' > {shlex.quote(str(YAI_HOME / 'run' / 'hot-state.json'))}")
run("yai hot status", check=False)
run(f"make uninstall-local PREFIX={shlex.quote(str(PREFIX))}")
run(f"test ! -e {shlex.quote(str(PREFIX / 'bin' / 'yai'))}")
run(f"test ! -e {shlex.quote(str(PREFIX / 'bin' / 'yaid'))}")